# AIDA — Natural Killer Cell Annotation Bias Analysis

This notebook investigates why CellTypist annotation F1 for **natural killer cell** (generic NK label) increases monotonically as SG_Chinese proportion in the reference rises from 0 % → 100 %.

Key result: Kendall τ = +0.514, p = 0.0288; mean F1 0.601 → 0.675 (+0.074).

In [3]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install scanpy celltypist -q
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/AIDA_project')

import scanpy as sc
import numpy as np
import pandas as pd
import json
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.metrics import f1_score, classification_report
import celltypist

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import scanpy as sc
import numpy as np
import pandas as pd
import json
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.metrics import f1_score, classification_report
import celltypist

In [4]:
# ── Load AIDA in backed mode + fix query donors (seed=42) ──────────────────
adata = sc.read_h5ad('AIDA.h5ad', backed='r')
print(adata)

sg_chinese_donors = adata.obs[
    adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese'
]['donor_id'].unique()

np.random.seed(42)
query_donors = np.random.choice(sg_chinese_donors, size=20, replace=False)
ref_candidate_donors = sg_chinese_donors[~np.isin(sg_chinese_donors, query_donors)]

all_other_donors = adata.obs[
    adata.obs['self_reported_ethnicity'].isin([
        'Japanese', 'Korean', 'Thai', 'Indian',
        'Singaporean Malay', 'Singaporean Indian'
    ])
]['donor_id'].unique()

print(f"Query donors       : {len(query_donors)}")
print(f"SG_Chinese ref pool: {len(ref_candidate_donors)}")
print(f"Other-ethnic pool  : {len(all_other_donors)}")


AnnData object with n_obs × n_vars = 1265624 × 35477 backed at 'AIDA.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'cell_number_loaded', 'institute', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'donor_BMI_at_collection', 'tissue_type', 'suspension_derivation_process', 'suspension_enriched_cell_types', 'cell_viability_percentage', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'pMito', 'NODG', 'nUMI', 'Country', 'Annotation_Level1', 'Annotation_Level2', 'Annotation_Level3', 'Annotation_Level4', 'Smoking Status', 'cell_type', 'assay', 'disease

## 1. Annotation quality of generic NK cells in the query set

`cell_type == 'natural killer cell'` is the **generic** NK label in CellTypeOntology —
cells that could not be resolved to a more specific subtype (CD16-dim or CD56-bright).
In the AIDA dataset, query-set generic NK cells are further classified at Level-4 as
`flagged_NK_low_exp` (60 %) and `flagged_platelet_sum` (40 %) — both QC-flagged categories
per AIDA STAR Methods — with no NK_unknown cells.
We first check what AIDA's own Annotation_Level4 says about these cells.

In [5]:
# ── NK annotation quality in query ─────────────────────────────────────────
query_mask = adata.obs['donor_id'].isin(query_donors)
nk_cells = adata.obs[query_mask & (adata.obs['cell_type'] == 'natural killer cell')]

print(f"Total generic NK cells in query: {len(nk_cells)}")
print()
print(nk_cells[['Annotation_Level1', 'Annotation_Level2',
                'Annotation_Level3', 'Annotation_Level4']].value_counts().head(20))

Total generic NK cells in query: 288

Annotation_Level1  Annotation_Level2  Annotation_Level3  Annotation_Level4   
NK                 NK                 NK                 flagged_NK_low_exp      174
                                                         flagged_platelet_sum    114
                                                         B_unknown                 0
                                                         CD4+_T_cm                 0
                                                         CD4+_T_cm_IFNhi           0
                                                         CD4+_T_cyt                0
                                                         CD4+_T_em                 0
                                                         CD4+_T_memory             0
                                                         CD4+_T_naive              0
                                                         CD4+_T_naive_IFNhi        0
                                  

## 2. Generic NK proportion: SG_Chinese vs. other ethnicities

Before interpreting the F1 trend we need to know whether SG_Chinese donors have
more or fewer generic NK cells than other groups.

In [6]:
# ── Generic NK proportion per ethnicity ────────────────────────────────────
nk_labels = [
    'natural killer cell',
    'CD16-positive, CD56-dim natural killer cell, human',
    'CD16-negative, CD56-bright natural killer cell, human'
]
other_ethnicities = ['Japanese', 'Korean', 'Thai', 'Indian',
                     'Singaporean Malay', 'Singaporean Indian']

sg_nk = adata.obs[
    (adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese') &
    (adata.obs['cell_type'].isin(nk_labels))
]
sg_total    = len(sg_nk)
sg_generic  = (sg_nk['cell_type'] == 'natural killer cell').sum()

other_nk = adata.obs[
    (adata.obs['self_reported_ethnicity'].isin(other_ethnicities)) &
    (adata.obs['cell_type'].isin(nk_labels))
]
other_total   = len(other_nk)
other_generic = (other_nk['cell_type'] == 'natural killer cell').sum()

print(f"SG_Chinese :  {sg_generic:>5}/{sg_total}  = {sg_generic/sg_total*100:.1f}% generic NK")
print(f"Other ethnic: {other_generic:>5}/{other_total} = {other_generic/other_total*100:.1f}% generic NK")
print(f"Difference  : {sg_generic/sg_total*100 - other_generic/other_total*100:.1f} pp")

SG_Chinese :   1284/19089  = 6.7% generic NK
Other ethnic: 21121/155748 = 13.6% generic NK
Difference  : -6.8 pp


In [7]:
# ── NK subtype composition in cell_type space ──────────────────────────────
print("SG_Chinese NK cell composition (cell_type labels):")
print((sg_nk['cell_type'].value_counts() / sg_total * 100).round(1)[:3])

print("\nOther ethnicities NK cell composition:")
print((other_nk['cell_type'].value_counts() / other_total * 100).round(1)[:3])

SG_Chinese NK cell composition (cell_type labels):
cell_type
CD16-positive, CD56-dim natural killer cell, human       90.3
natural killer cell                                       6.7
CD16-negative, CD56-bright natural killer cell, human     3.0
Name: count, dtype: float64

Other ethnicities NK cell composition:
cell_type
CD16-positive, CD56-dim natural killer cell, human       83.1
natural killer cell                                      13.6
CD16-negative, CD56-bright natural killer cell, human     3.4
Name: count, dtype: float64


## 3. Annotation_Level4 composition: query, SG_Chinese ref, other ethnicities

The generic NK cells (`cell_type == 'natural killer cell'`) in all ethnicities share
the same Level-1 label (`NK`), but their Level-4 sub-labels differ substantially.

In [8]:
# ── NK Level-4 composition: query vs. reference SG_Chinese vs. others ──────
query_nk_obs = adata.obs[
    adata.obs['donor_id'].isin(query_donors) &
    (adata.obs['cell_type'] == 'natural killer cell')
]

ref_sgchinese_nk = adata.obs[
    (adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese') &
    (~adata.obs['donor_id'].isin(query_donors)) &
    (adata.obs['cell_type'] == 'natural killer cell')
]

other_nk_obs = adata.obs[
    adata.obs['self_reported_ethnicity'].isin(other_ethnicities) &
    (adata.obs['cell_type'] == 'natural killer cell')
]

def nk_level4_summary(obs_df, label):
    total  = len(obs_df)
    counts = obs_df['Annotation_Level4'].value_counts()
    pct    = (counts / total * 100).round(1)
    df     = pct[pct > 0].to_frame(name='% of NK cells')
    df['cell count'] = counts[pct > 0]
    print(f"\n== {label} (total NK = {total}) ==")
    print(df.to_string())

nk_level4_summary(query_nk_obs,     "Query SG_Chinese NK")
nk_level4_summary(ref_sgchinese_nk, "Reference SG_Chinese NK")
nk_level4_summary(other_nk_obs,     "Other ethnicities NK (pooled)")


== Query SG_Chinese NK (total NK = 288) ==
                      % of NK cells  cell count
Annotation_Level4                              
flagged_NK_low_exp             60.4         174
flagged_platelet_sum           39.6         114

== Reference SG_Chinese NK (total NK = 996) ==
                      % of NK cells  cell count
Annotation_Level4                              
flagged_NK_low_exp             68.2         679
flagged_platelet_sum           31.8         317

== Other ethnicities NK (pooled) (total NK = 21121) ==
                      % of NK cells  cell count
Annotation_Level4                              
NK_unknown                     43.6        9199
flagged_platelet_sum           40.5        8553
flagged_NK_low_exp             16.0        3369


In [19]:
# ── Key contrast summary ───────────────────────────────────────────────────
def composition_breakdown(obs_df, label):
    total  = len(obs_df)
    counts = obs_df['Annotation_Level4'].value_counts()
    pct    = (counts / total * 100).round(1)

    nk_unknown_pct = pct.get('NK_unknown', 0.0)
    low_exp_pct    = pct.get('flagged_NK_low_exp', 0.0)
    platelet_pct   = pct.get('flagged_platelet_sum', 0.0)

    print(f"\n== {label} (n={total}) ==")
    print(f"  NK_unknown          : {nk_unknown_pct:5.1f}%")
    print(f"  flagged_NK_low_exp  : {low_exp_pct:5.1f}%")
    print(f"  flagged_platelet_sum: {platelet_pct:5.1f}%")

composition_breakdown(query_nk_obs,     "Query SG_Chinese NK")
composition_breakdown(ref_sgchinese_nk, "Reference SG_Chinese NK")
composition_breakdown(other_nk_obs,     "Other ethnicities NK (pooled)")

print("""
Key contrast:
  SG_Chinese: 0% NK_unknown, ~68% flagged_NK_low_exp
  Others:    ~44% NK_unknown, ~16% flagged_NK_low_exp

Interpretation: the F1 gain with more SG_Chinese reference reflects
transcriptomically better-matched training examples for the query's generic NK cells.
CellTypist trains only on Level-1 labels ("natural killer cell") and does not see
Level-4 annotations. However, SG_Chinese reference NK cells are predominantly
flagged_NK_low_exp at Level-4 — the same transcriptomic profile as the query —
whereas other-ethnicity reference NK cells are predominantly NK_unknown, a
distinct transcriptomic subtype (lower canonical NK marker expression relative
to NK_unknown, higher MALAT1/NEAT1 per AIDA STAR Methods).
This is a reference composition effect at the transcriptomic level.
""")


== Query SG_Chinese NK (n=288) ==
  NK_unknown          :   0.0%
  flagged_NK_low_exp  :  60.4%
  flagged_platelet_sum:  39.6%

== Reference SG_Chinese NK (n=996) ==
  NK_unknown          :   0.0%
  flagged_NK_low_exp  :  68.2%
  flagged_platelet_sum:  31.8%

== Other ethnicities NK (pooled) (n=21121) ==
  NK_unknown          :  43.6%
  flagged_NK_low_exp  :  16.0%
  flagged_platelet_sum:  40.5%

Key contrast:
  SG_Chinese: 0% NK_unknown, ~68% flagged_NK_low_exp
  Others:    ~44% NK_unknown, ~16% flagged_NK_low_exp

Interpretation: the F1 gain with more SG_Chinese reference reflects
transcriptomically better-matched training examples for the query's generic NK cells.
CellTypist trains only on Level-1 labels ("natural killer cell") and does not see
Level-4 annotations. However, SG_Chinese reference NK cells are predominantly
flagged_NK_low_exp at Level-4 — the same transcriptomic profile as the query —
whereas other-ethnicity reference NK cells are predominantly NK_unknown, a
distinct 

## 4. F1 / precision / recall across SG_Chinese proportions

Results are loaded from `all_reports.json` produced by the full-curve experiment
(36 replicates × 11 proportions 0 %–100 %).

In [10]:
# ── Load results ───────────────────────────────────────────────────────────
with open('all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows = []
for entry in all_reports:
    if 'natural killer cell' in entry['report']:
        r = entry['report']['natural killer cell']
        rows.append({
            'proportion': entry['proportion'],
            'repeat':     entry['repeat'],
            'precision':  r['precision'],
            'recall':     r['recall'],
            'f1':         r['f1-score']
        })

df_nk = pd.DataFrame(rows)
summary = df_nk.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print(summary)

           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.704  0.050  0.527  0.064  0.601  0.050
0.1            0.719  0.049  0.577  0.066  0.638  0.049
0.2            0.703  0.040  0.587  0.064  0.638  0.047
0.3            0.732  0.035  0.581  0.044  0.647  0.031
0.4            0.725  0.042  0.612  0.031  0.663  0.026
0.5            0.743  0.029  0.623  0.038  0.677  0.028
0.6            0.718  0.054  0.610  0.041  0.659  0.038
0.7            0.711  0.043  0.598  0.028  0.649  0.020
0.8            0.721  0.027  0.606  0.022  0.658  0.020
0.9            0.723  0.029  0.601  0.022  0.656  0.017
1.0            0.745  0.016  0.616  0.015  0.675  0.011


In [11]:
# ── Replicate consistency: how often does 100% > 0%? ─────────────────────
pivot = df_nk.pivot(index='repeat', columns='proportion', values='f1')
pivot['diff_100_vs_0'] = pivot[1.0] - pivot[0.0]

print(pivot[['diff_100_vs_0']].describe().round(3))
print()
print(f"Replicates where 100% > 0%: {(pivot['diff_100_vs_0'] > 0).sum()} / {len(pivot)}")

proportion  diff_100_vs_0
count              33.000
mean                0.075
std                 0.053
min                -0.028
25%                 0.033
50%                 0.065
75%                 0.134
max                 0.157

Replicates where 100% > 0%: 32 / 36


In [12]:
# ── Monotonicity test (Kendall τ) ──────────────────────────────────────────
props  = summary.index.tolist()
f1vals = summary['f1']['mean'].tolist()

tau, p = stats.kendalltau(props, f1vals)
print(f"Kendall τ (NK F1 vs. proportion): {tau:.3f}, p = {p:.4f}")
print(f"Mean F1 range: {min(f1vals):.3f} → {max(f1vals):.3f}")
print(f"Total gain (100% vs 0%): {f1vals[-1] - f1vals[0]:+.3f}")

Kendall τ (NK F1 vs. proportion): 0.514, p = 0.0288
Mean F1 range: 0.601 → 0.677
Total gain (100% vs 0%): +0.074


## 5. Misclassification analysis (10 runs, 0% vs. 100% reference)

We re-run CellTypist at 0 % and 100 % SG_Chinese reference to see what the
generic NK cells get mislabelled as.

> **Note:** this cell re-runs the model (10 × 2 = 20 CellTypist jobs). Skip if
> results are already saved.

In [15]:
def run_celltypist(adata, query_donors, other_donors, ref_candidate_donors,
                   sg_chinese_proportion, n_ref_total=60):
    n_chinese = int(n_ref_total * sg_chinese_proportion)
    n_others  = n_ref_total - n_chinese

    chinese_ref = np.random.choice(ref_candidate_donors, size=n_chinese, replace=False) \
                  if n_chinese > 0 else []
    other_ref   = np.random.choice(other_donors, size=n_others, replace=False)
    all_ref_donors = list(chinese_ref) + list(other_ref)

    ref_adata = adata[adata.obs['donor_id'].isin(all_ref_donors)].to_memory()
    sc.pp.highly_variable_genes(ref_adata, n_top_genes=2000)
    ref_adata = ref_adata[:, ref_adata.var['highly_variable']]

    model = celltypist.train(ref_adata, labels='cell_type', n_jobs=-1,
                             with_mean=False, check_expression=False)

    query = adata[adata.obs['donor_id'].isin(query_donors)].to_memory()
    query = query[:, ref_adata.var_names].copy()
    if sp.issparse(query.X):
        query.X = query.X.toarray()

    predictions  = celltypist.annotate(query, model=model)
    true_labels  = query.obs['cell_type']
    pred_labels  = predictions.predicted_labels['predicted_labels']
    f1 = f1_score(true_labels, pred_labels, average='macro')
    return f1, predictions, query

In [16]:
# ── Misclassification accumulation over 10 runs ───────────────────────────
n_runs = 10
nk_misclass = {'0%': {}, '100%': {}}

for i in range(n_runs):
    print(f"Run {i+1}/{n_runs} ...")
    for prop, key in [(0.0, '0%'), (1.0, '100%')]:
        f1, pred, query = run_celltypist(
            adata, query_donors, all_other_donors,
            ref_candidate_donors, sg_chinese_proportion=prop)
        true_lab = query.obs['cell_type']
        pred_lab = pred.predicted_labels['predicted_labels']
        nk_mask  = true_lab == 'natural killer cell'
        for label, count in pred_lab[nk_mask].value_counts().items():
            nk_misclass[key][label] = nk_misclass[key].get(label, 0) + count

df_0   = pd.Series(nk_misclass['0%']).sort_values(ascending=False)
df_100 = pd.Series(nk_misclass['100%']).sort_values(ascending=False)

print("\n0% SG_Chinese reference — NK predicted as (10-run total):")
print(df_0[df_0 > 0])

print("\n100% SG_Chinese reference — NK predicted as (10-run total):")
print(df_100[df_100 > 0])

Run 1/10 ...


Run 2/10 ...


Run 3/10 ...


Run 4/10 ...


Run 5/10 ...


...

Run 7/10 ...


Run 8/10 ...


Run 9/10 ...


Run 10/10 ...



0% SG_Chinese reference — NK predicted as (10-run total):
natural killer cell                                      1509
CD16-positive, CD56-dim natural killer cell, human        699
CD8-positive, alpha-beta cytotoxic T cell                 226
CD8-positive, alpha-beta T cell                           135
T cell                                                     94
gamma-delta T cell                                         83
CD8-positive, alpha-beta memory T cell                     79
CD4-positive, alpha-beta cytotoxic T cell                  27
mucosal-associated invariant T cell                        15
CD16-negative, CD56-bright natural killer cell, human       9
CD4-positive, alpha-beta T cell                             2
platelet                                                    1
regulatory T cell                                           1
dtype: int64

100% SG_Chinese reference — NK predicted as (10-run total):
natural killer cell                                      1729

## Summary

| | 0% SG_Chinese ref | 100% SG_Chinese ref |
|---|---|---|
| NK correctly labelled (10 runs) | 1509 | 1729 |
| Largest misclass | CD16+ NK 699, CD8+ cytotoxic T 226 | CD16+ NK 513, T cell 233 |
| Mean F1 | 0.601 | 0.675 |

**Mechanism:** The query's generic NK cells are almost entirely `flagged_NK_low_exp`
at Level-4 (60 %) and `flagged_platelet_sum` (40 %), with **0 % NK_unknown**.
`flagged_NK_low_exp` is a QC-flagged cluster defined in the AIDA STAR Methods as having
(a) low range of NODG values, (b) low expression of canonical NK marker genes
(e.g., NKG7, GNLY, KLRD1), and (c) heightened MALAT1/NEAT1 expression.
Other-ethnicity NK cells contain 44 % NK_unknown, providing a poor exemplar match.
Adding more SG_Chinese donors to the reference brings in generic NK training examples
that are transcriptomically similar to the query's NK cells (both predominantly
flagged_NK_low_exp at Level-4), directly improving recall.
Note: CellTypist operates at Level-1 only; Level-4 labels are used here
as a diagnostic tool to characterize reference composition.